In [ ]:
!pip -q install ultralytics huggingface_hub
import torch;print('GPU:',torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE')

In [ ]:
# Data: cute-face/bike-helmet-dataset (HF card: cc-by-4.0; embedded README says Public Domain; original author Syed Salman Reza via Roboflow). COCO train/valid/test folders ONLY (not google_search_helmets).
from huggingface_hub import snapshot_download, hf_hub_download
import json,os,shutil
snapshot_download('cute-face/bike-helmet-dataset',repo_type='dataset',allow_patterns=['train/*','valid/*','test/*'],local_dir='bh')
for s in ['train','valid','test']:
    j=json.load(open(f'bh/{s}/_annotations.coco.json')); os.makedirs(f'y/{s}/images',exist_ok=True); os.makedirs(f'y/{s}/labels',exist_ok=True)
    ims={i['id']:i for i in j['images']}; lab={}
    for a in j['annotations']:
        if a['category_id'] not in (1,2): continue
        im=ims[a['image_id']]; x,y,w,h=a['bbox']; c=0 if a['category_id']==1 else 1
        lab.setdefault(a['image_id'],[]).append(f"{c} {(x+w/2)/im['width']:.6f} {(y+h/2)/im['height']:.6f} {w/im['width']:.6f} {h/im['height']:.6f}")
    for iid,im in ims.items():
        shutil.copy(f"bh/{s}/{im['file_name']}",f"y/{s}/images/{im['file_name']}"); open(f"y/{s}/labels/{os.path.splitext(im['file_name'])[0]}.txt",'w').write("\n".join(lab.get(iid,[])))
    print(s,len(ims))
open('y/data.yaml','w').write("path: /content/y\ntrain: train/images\nval: valid/images\ntest: test/images\nnames:\n  0: helmet\n  1: no_helmet\n")
shutil.copy(hf_hub_download('Smolry/Helmet-classifer','helmet.pt'),'base.pt')

In [ ]:
from ultralytics import YOLO
def ev(path,split):
    r=YOLO(path).val(data='y/data.yaml',split=split,imgsz=640,conf=0.25,verbose=False,plots=False)
    return {r.names[i]:dict(P=round(float(r.box.p[i]),3),R=round(float(r.box.r[i]),3),mAP50=round(float(r.box.ap50[i]),3)) for i in r.names}
print('BASELINE test',ev('base.pt','test')); print('BASELINE valid',ev('base.pt','val'))

In [ ]:
YOLO('base.pt').train(data='y/data.yaml',epochs=40,imgsz=640,batch=16,patience=10,project='runs',name='ft',plots=False,verbose=False)
print('AFTER test',ev('runs/ft/weights/best.pt','test')); print('AFTER valid',ev('runs/ft/weights/best.pt','val'))